# 10 — W2 CMQR + StateTracker on the dev set

Per RecSys_Challenge_Plan §A2 + §4 (W2 gate). Runs the full dev-set pipeline (1000 sessions × 8 turns) with CMQR enabled, scores via the official evaluator, and compares against the exp-021 champion.

**What this notebook does:**
1. Mounts Drive (HF cache + experiments cache survive runtime resets).
2. Clones `fresh-model` branch, installs deps + vLLM.
3. Runs `run_inference_devset.py --tid 100-cmqr-state-qwen15b-devset` with the new YAML config.
4. Runs `music-crs-evaluator/evaluate_devset.py` for nDCG@1/10/20 + diversity terms.
5. Compares vs exp-021 anchor (nDCG@10 = 0.0784).
6. Saves both prediction.json and the scores to Drive.

**W2 gate:** dev nDCG@10 ≥ 0.0834 (champion +0.005), OR diagnose why.
**Wall time on A100:** ~20–30 min including model loads, full devset inference, and eval.

In [ ]:
# HF auth — set HF_TOKEN in Colab Secrets (🔑 icon in the left sidebar).
#
# Steps:
#   1) Click the 🔑 key icon in the Colab sidebar (or Tools → Secrets).
#   2) Click "+ Add new secret".
#   3) Name: HF_TOKEN  (exact, case-sensitive).
#   4) Value: your User Access Token from https://huggingface.co/settings/tokens
#      with WRITE scope (read-only fails at trainer.push_to_hub).
#   5) Toggle "Notebook access" ON for this notebook.
import os
from google.colab import userdata

try:
    HF_TOKEN = userdata.get('HF_TOKEN')
except (userdata.SecretNotFoundError, Exception) as e:
    raise SystemExit(
        f'❌ HF_TOKEN secret not found in Colab ({e!r}).\n'
        f'   1) Open the 🔑 Secrets pane in the left sidebar.\n'
        f'   2) Add a secret named exactly `HF_TOKEN` with your User Access Token.\n'
        f'   3) Toggle "Notebook access" ON for this notebook.'
    )

os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HUGGINGFACE_HUB_TOKEN'] = HF_TOKEN

from huggingface_hub import whoami
try:
    user = whoami(token=HF_TOKEN)
    print(f'✓ HF auth ok — logged in as {user["name"]}')
except Exception as e:
    raise SystemExit(
        f'❌ HF auth failed: {e!r}\n'
        f'   Check the token at https://huggingface.co/settings/tokens '
        f'(needs WRITE scope for push_to_hub).'
    )

In [ ]:
# 1) GPU check.
!nvidia-smi | head -20

In [ ]:
# 2) Clone fresh-model branch.
BRANCH = 'fresh-model'
!rm -rf /content/recsys2026
!git clone -b {BRANCH} https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026
%cd /content/recsys2026
print('\n=== CODE VERSION CHECK ===')
!git log -1 --pretty=format:'commit:  %h%ndate:    %ai%nsubject: %s'
print()

In [ ]:
# 2b) Mount Drive + persistent caches. Same convention as 02_state_tracker_prototype.ipynb.
import os, shutil
from google.colab import drive

try:
    drive.mount('/content/drive')
except Exception as e:
    print(f'first mount attempt failed: {e}\nretrying with force_remount=True ...')
    try:
        drive.flush_and_unmount()
    except Exception:
        pass
    drive.mount('/content/drive', force_remount=True)

assert os.path.isdir('/content/drive/MyDrive'), (
    'Drive mount failed. Complete the OAuth popup and retry.')

DRIVE_BASE = '/content/drive/MyDrive/recsys2026-cache'
for d in [f'{DRIVE_BASE}/hf_datasets', f'{DRIVE_BASE}/experiments_cache',
          f'{DRIVE_BASE}/cmqr_dev']:
    os.makedirs(d, exist_ok=True)

os.environ['HF_DATASETS_CACHE'] = f'{DRIVE_BASE}/hf_datasets'
%env HF_DATASETS_CACHE={DRIVE_BASE}/hf_datasets

EXPECTED_CACHE = '/content/recsys2026/music-crs-baselines/experiments/cache'
os.makedirs(os.path.dirname(EXPECTED_CACHE), exist_ok=True)
if os.path.exists(EXPECTED_CACHE) and not os.path.islink(EXPECTED_CACHE):
    shutil.rmtree(EXPECTED_CACHE)
if not os.path.islink(EXPECTED_CACHE):
    os.symlink(f'{DRIVE_BASE}/experiments_cache', EXPECTED_CACHE)

print(f'datasets cache  : {os.environ["HF_DATASETS_CACHE"]}')
print(f'experiments dir : {EXPECTED_CACHE} -> {os.readlink(EXPECTED_CACHE)}')

In [ ]:
# 3) Install deps. The W2 config uses use_vllm=true so we need vLLM.
!pip install -q --upgrade transformers datasets pandas tqdm omegaconf bm25s sentence-transformers
!pip install -q --upgrade vllm
!python -c 'import torch, transformers, vllm; print("torch", torch.__version__, "cuda", torch.cuda.is_available(), "vllm", vllm.__version__)'

In [ ]:
# 4) Pytest gate — confirm CMQR module + integration tests pass before burning A100 time.
!cd /content/recsys2026 && python -m pytest tests/test_cmqr.py tests/test_state_tracker.py tests/test_reward_fns.py -q

In [ ]:
# 5) Run inference on the full devset with CMQR enabled.
# The config 100-cmqr-state-qwen15b-devset.yaml has:
#   - use_state_tracker: true  (required by CMQR)
#   - use_cmqr: true           (the W2 lever)
#   - retrieval_topk: 100      (plan §A3 — wider pool for fusion)
#   - use_vllm: true           (5-10x throughput on A100)
TID = '100-cmqr-state-qwen15b-devset'
BATCH_SIZE = 16
DEVICE = 'cuda'
ATTN = 'sdpa'  # CUDA-only; do not use flash_attention_2 (compile required)

!nvidia-smi --query-gpu=memory.total,memory.used,memory.free --format=csv
!cd /content/recsys2026/music-crs-baselines && python run_inference_devset.py \
    --tid {TID} --batch_size {BATCH_SIZE} --device {DEVICE} --attn_implementation {ATTN}

In [ ]:
# 6) Score with the official evaluator.
!cd /content/recsys2026/music-crs-evaluator && python evaluate_devset.py --tid {TID}

In [ ]:
# 7) Read scores + check the W2 gate.
import json
from pathlib import Path

scores_path = Path(f'/content/recsys2026/music-crs-evaluator/exp/scores/devset/{TID}.json')
with scores_path.open() as f:
    scores = json.load(f)

ndcg10 = scores.get('ndcg@10', 0.0)
ndcg20 = scores.get('ndcg@20', 0.0)
ndcg1 = scores.get('ndcg@1', 0.0)
catdiv = scores.get('catalog_diversity', 0.0)
lexdiv = scores.get('lexical_diversity', 0.0)

# Champion anchor (exp-020 devset / exp-021 Blind-A) — recovered from
# git a81696b documents/submissions_log.md.
CHAMPION_NDCG10 = 0.0784
CHAMPION_NDCG20 = 0.0995
GATE_THRESHOLD = CHAMPION_NDCG10 + 0.005

print('=' * 60)
print('W2 CMQR — DEV SET RESULTS')
print('=' * 60)
print(f'  nDCG@1  : {ndcg1:.4f}')
print(f'  nDCG@10 : {ndcg10:.4f}  (champion {CHAMPION_NDCG10}, gate ≥ {GATE_THRESHOLD})')
print(f'  nDCG@20 : {ndcg20:.4f}  (champion {CHAMPION_NDCG20})')
print(f'  CatDiv  : {catdiv:.4f}')
print(f'  LexDiv  : {lexdiv:.4f}')
print()
delta = ndcg10 - CHAMPION_NDCG10
print(f'  Δ vs champion nDCG@10 : {delta:+.4f}')
print()
print('GATE (plan §4 W2 row):')
if ndcg10 >= GATE_THRESHOLD:
    print(f'  PASS   {ndcg10:.4f} ≥ {GATE_THRESHOLD} — proceed to W3 (ProRank reranker)')
elif ndcg10 >= CHAMPION_NDCG10:
    print(f'  WEAK PASS  {ndcg10:.4f} ≥ {CHAMPION_NDCG10} but < gate; investigate')
    print('  Likely diagnoses: rewrites too redundant, RRF weights wrong, state too vague')
else:
    print(f'  FAIL   {ndcg10:.4f} < {CHAMPION_NDCG10} — CMQR regressed retrieval')
    print('  Investigate: are rewrites preserving the original intent?')
    print('  Try: cmqr_n_rewrites=2 (less divergence), cmqr_topk_per_rewrite=100 (wider per-stream)')
print('=' * 60)

In [ ]:
# 8) Save artifacts to Drive (predictions + scores).
import shutil
pred_src = f'/content/recsys2026/music-crs-baselines/exp/inference/devset/{TID}.json'
scores_src = f'/content/recsys2026/music-crs-evaluator/exp/scores/devset/{TID}.json'
drive_dst = f'{DRIVE_BASE}/cmqr_dev'
shutil.copy(pred_src, f'{drive_dst}/{TID}.json')
shutil.copy(scores_src, f'{drive_dst}/scores_{TID}.json')
print(f'predictions: {drive_dst}/{TID}.json')
print(f'scores:      {drive_dst}/scores_{TID}.json')
!ls -lh {drive_dst}/

In [ ]:
# 9) Inspect a few sample predictions to eyeball quality.
import json
with open(pred_src) as f:
    preds = json.load(f)
print(f'total rows: {len(preds)}')
for r in preds[:3]:
    print()
    print(f"session={r['session_id'][:8]}  turn={r['turn_number']}")
    print(f"  predicted_track_ids[:5]: {r['predicted_track_ids'][:5]}")
    print(f"  predicted_response: {r['predicted_response'][:200]!r}")

In [ ]:
# 10) CMQR diagnostics — sample the cached rewrites.
import os, json
from pathlib import Path
cmqr_cache = Path(f'{DRIVE_BASE}/experiments_cache/cmqr')
if not cmqr_cache.exists():
    cmqr_cache = Path('/content/recsys2026/music-crs-baselines/experiments/cache/cmqr')
files = list(cmqr_cache.glob('*.json'))[:5]
print(f'cmqr cache files: {len(list(cmqr_cache.glob("*.json")))}  (showing first 5)')
for fp in files:
    with fp.open() as f:
        c = json.load(f)
    print(f'\n{fp.name}')
    print(f'  original: {c["original_query"][:80]!r}')
    for i, r in enumerate(c['rewrites'], 1):
        print(f'  rewrite {i}: {r}')

## After the run

If `nDCG@10 >= 0.0834` (PASS):
- Update `documents/RecSys_Challenge_Plan.md` with the W2 result.
- Append a row to `documents/submissions_log.md` (`[dev-local]` tag).
- Proceed to **W3** — A5 ProRank listwise reranker on top of the CMQR top-100.

If `nDCG@10 < 0.0784` (regression):
- Sample failed turns (gold not in predicted top-100) — diagnose whether rewrites are too divergent (preserve_original=True option) or too redundant (push N_rewrites=6 with diversity penalty).
- Try `cmqr_n_rewrites=2` (less LM cost, more conservative).
- If still regressed, fall back to vanilla wRRF + StateTracker only (drop CMQR) and continue to W3 — A5 reranker may still lift the floor.

If WEAK PASS (above champion but below +0.005 gate):
- Ship to W3 anyway — the reranker may close the gap.
- Keep the artifact at Drive for ablation against other CMQR weight configurations.